# Proyecto: 

### 1. Justificación del problema

La empresa M&S cuenta con un aplicativo web para la gestión de productos químicos. En la base de datos se almacena información asociada a los productos a partir de las Fichas de Datos de Seguridad (FDS), las cuales funcionan como la hoja de vida del producto.

Debido a la gran variabilidad en la forma en que estos productos pueden ser nombrados, actualmente existe un problema significativo en la gestión del catálogo. Se estima que aproximadamente el 7 % de los productos registrados se encuentran duplicados, lo que genera inconsistencias en la base de datos y aumenta los tiempos necesarios para consolidar inventarios provenientes de diferentes clientes.

Los clientes que utilizan el aplicativo registran los productos de acuerdo con los nombres internos utilizados dentro de sus organizaciones. Como resultado, un mismo producto puede aparecer en el sistema bajo múltiples variantes de nombre, diferencias en abreviaturas, idioma, formato o referencia comercial.

Por ejemplo, un mismo producto podría registrarse como:
•	NaOH 0.1N Cleaning Solution
•	0.1 N Sodium Hydroxide Solution
•	NaOH 0.1N cuvette cleaning solution

Adicionalmente, diferentes empresas pueden referirse a un mismo producto con denominaciones distintas. Por ejemplo, un adhesivo puede ser llamado pegante, pega, adhesivo industrial o cemento de contacto, dependiendo del contexto organizacional o regional.

Actualmente, el cruce entre los productos registrados por los clientes y los productos existentes en el catálogo se realiza en gran parte de manera manual, lo que implica:
•	altos tiempos de revisión
•	mayor probabilidad de duplicidad en los registros
•	inconsistencias en la base de datos
•	dificultades en la búsqueda y recuperación de información.

Ante este contexto, resulta relevante desarrollar mecanismos automáticos que permitan identificar qué productos del catálogo existente son más similares a un nuevo registro ingresado al sistema, utilizando la información disponible como el nombre del producto, sus sinónimos, su uso y el fabricante.

Un sistema que sugiera coincidencias probables permitiría:
•	reducir la creación de duplicados
•	mejorar la calidad y consistencia del catálogo de productos
•	facilitar la búsqueda y recuperación de información dentro del sistema
•	optimizar procesos de gestión de inventarios químicos.

Por esta razón, se propone explorar el uso de técnicas de procesamiento de lenguaje natural (NLP) basadas en embeddings neuronales que permitan representar semánticamente los productos y calcular similitudes entre ellos dentro del catálogo.

### 2. Objetivo
Desarrollar un modelo basado en embeddings semánticos que, utilizando la información textual disponible de los productos —nombre del producto, sinónimos, uso y fabricante— identifique los productos más similares dentro del catálogo existente y devuelva un ranking de coincidencias probables, facilitando el proceso de validación y evitando la creación de registros duplicados.

### 3. Metodología
**3.1 Estructura inicial de los datos**

El dataset de productos se divide en dos subconjuntos dependiendo de la disponibilidad de información adicional.
df_con_sin_o_uso
Productos que cuentan con información en las columnas de uso o sinónimos.
Tamaño aproximado:
(50621, 5)
df_sin_sin_o_uso **## ESTA DIVISIÓN NO LA IMPLEMENTE, PERO SERIA INTERESANTE HACERLO**
Productos que no cuentan con información adicional en esas columnas.
Tamaño aproximado:
(13874, 5)
La motivación de esta separación nace del hecho de que los productos que cuentan con más información contextual permiten construir representaciones semánticas más completas, lo que potencialmente mejora el desempeño del modelo.
Sin embargo, ambos subconjuntos forman parte del mismo catálogo y pueden ser utilizados en el sistema final.

**3.2 Limpieza y normalización de texto**

Con el fin de preparar los datos para su utilización en modelos de procesamiento de lenguaje natural, se aplicaron diversos procedimientos de limpieza y normalización del texto.

Las principales transformaciones realizadas fueron:
•	conversión del texto a minúsculas
•	eliminación de acentos y caracteres especiales innecesarios
•	eliminación de espacios duplicados
•	reemplazo de valores nulos por cadenas vacías
•	normalización de símbolos y formatos.

Estas transformaciones permiten reducir la variabilidad innecesaria en los datos y facilitan el proceso de tokenización posterior.

**3.3 Construcción de la representación textual**

Para representar cada producto de manera más completa, se construyó una columna consolidada de texto combinando la información disponible en las diferentes variables del dataset.
La representación textual final se construye concatenando:
•	nombre del producto
•	sinónimos
•	uso del producto
•	fabricante

Esto permite que el modelo aproveche simultáneamente todas las fuentes de información disponibles, enriqueciendo la representación semántica del producto.
La inclusión del fabricante resulta especialmente útil para reducir falsos positivos, ya que muchos productos pueden tener nombres similares, pero pertenecer a fabricantes distintos.

**3.4 Selección del modelo**

El problema planteado consiste en identificar productos similares dentro de un catálogo a partir de múltiples atributos textuales. En términos metodológicos, esto corresponde a una tarea de búsqueda semántica y recuperación de información (semantic retrieval).
A diferencia de los problemas clásicos de clasificación, donde el objetivo es asignar una única etiqueta a cada observación, en este caso el sistema debe ser capaz de recuperar los productos más similares a un registro dado y ordenarlos de acuerdo con su nivel de similitud.
Para este propósito se seleccionó el algoritmo Word2Vec, un modelo de embeddings que utiliza una red neuronal para aprender representaciones vectoriales de las palabras a partir de su contexto dentro de un corpus textual.
Este tipo de modelos permite mapear palabras a un espacio vectorial continuo, donde palabras con significado similar tienden a ubicarse cerca entre sí. Por ejemplo, términos como pegante y adhesivo pueden quedar próximos en este espacio, incluso si no aparecen exactamente iguales en el texto.

**3.5 Desarrollo del modelo**

El sistema desarrollado sigue el siguiente pipeline:
1.	Tokenización del texto consolidado
El texto consolidado de cada producto se divide en tokens o palabras individuales.
2.	Entrenamiento del modelo Word2Vec
Se entrena un modelo Word2Vec sobre el corpus completo de productos para aprender embeddings semánticos de las palabras presentes en el catálogo.
3.	Representación vectorial de cada producto
Cada producto se representa como el promedio de los embeddings de las palabras que lo componen, generando así un vector numérico que captura su significado semántico.
4.	Cálculo de similitud entre productos
Para comparar productos se utiliza la similitud coseno, una métrica ampliamente utilizada para medir la similitud entre vectores en espacios de alta dimensión.
5.	Recuperación de productos similares

Dado un nuevo producto de entrada, el sistema:
•	genera su representación vectorial
•	calcula su similitud con todos los productos del catálogo
•	devuelve los Top-k productos más similares.

**3.6 Evaluación del modelo: ##HAY QUE REVISARLO, NO ESTOY SEGURA DE COMO HACERLO, IMPLEMENTE ESTAS DOS, PERO EL RECALL SIEMPRE DA 1 POR QUE SIEMPRE SE ENCUENTRA A SI MISMO, Y LA PRECISIÓN TOP-K SIEMPRE DA CERCANO AL 20% POR LO MISMO.**

La evaluación del sistema se realiza analizando la capacidad del modelo para recuperar productos similares dentro del catálogo.
Se utilizaron principalmente métricas basadas en ranking, tales como:

•	Recall_k: mide si el producto correcto aparece dentro de los primeros k resultados.
•	Precisión en Top-k: mide la proporción de coincidencias correctas dentro de los resultados devueltos.
Adicionalmente, se evaluó el comportamiento del modelo mediante ejemplos reales de consulta provenientes del catálogo de productos.

In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd

### Carga de datos

In [2]:
df = pd.read_excel("listado productos.xlsx").iloc[:, 1:]
df

,id,nombre_producto,sinonimos,uso,fabricante
0,30818,0 1 m dtt,superscript iv first strand synthesis system |...,para uso en investigacion unicamente,thermofisher scientific
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,NaN,beckman coulter
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare
3,42583,0 45 m 47mm white gridded s pak filter membran...,NaN,investigacion y analisis bioquimicos,merck
4,56156,0 5 medios de crecimiento b de postgate modifi...,NaN,NaN,grupo volante
...,...,...,...,...,...
64490,35967,zytiga,zytiga abiraterone acetate abiraterone acetate...,producto farmaceutico terminado grupo farmacot...,janssen
64491,1360,zytokil 10mg 5ml,doxorubicina,no registra,pisa farmaceutica
64492,26325,zz76 01009 citrus passion,NaN,NaN,ungerer & company
64493,62579,zzliteprop prime 108 alt code l426802 00,NaN,NaN,grupo transmerquim


### Limpieza y normalización

In [3]:
# 3. Estandarizar nombres de columnas
df.columns = [c.strip().lower() for c in df.columns]
print("Columnas:", df.columns.tolist())

Columnas: ['id', 'nombre_producto', 'sinonimos', 'uso', 'fabricante']


In [4]:
# 4. Revisar nulos
print("Valores nulos por columna:")
print(df.isna().sum())

Valores nulos por columna:
id                     0
nombre_producto        0
sinonimos          35881
uso                16279
fabricante          1948
dtype: int64


In [5]:
# 5. Funciones de limpieza de texto
def quitar_acentos(texto: str) -> str:
    """
    Elimina tildes y caracteres diacríticos.
    """
    texto = unicodedata.normalize("NFKD", texto)
    return "".join([c for c in texto if not unicodedata.combining(c)])


def limpiar_texto(texto):
    """
    Limpieza y normalización básica para columnas textuales.
    """
    # Manejo de nulos
    if pd.isna(texto):
        return ""
    
    # Asegurar string
    texto = str(texto)
    
    # Minúsculas
    texto = texto.lower()
    
    # Quitar acentos
    texto = quitar_acentos(texto)
    
    # Reemplazar separadores comunes por espacio
    texto = texto.replace("|", " ")
    texto = texto.replace("/", " ")
    texto = texto.replace("\\", " ")
    texto = texto.replace("-", " ")
    texto = texto.replace("_", " ")
    
    # Eliminar caracteres especiales, pero conservar letras, números y espacios
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    
    # Eliminar espacios duplicados
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto

In [6]:
# 6. Aplicar limpieza a columnas textuales
columnas_texto = ["nombre_producto", "sinonimos", "uso", "fabricante"]

for col in columnas_texto:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto)

print("Ejemplo después de limpieza:")
display(df.head())

Ejemplo después de limpieza:


,id,nombre_producto,sinonimos,uso,fabricante
0,30818,0 1 m dtt,superscript iv first strand synthesis system k...,para uso en investigacion unicamente,thermofisher scientific
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,,beckman coulter
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare
3,42583,0 45 m 47mm white gridded s pak filter membran...,,investigacion y analisis bioquimicos,merck
4,56156,0 5 medios de crecimiento b de postgate modifi...,,,grupo volante


In [8]:
# 7. Revisar registros vacíos en columnas principales

for col in columnas_texto:
    if col in df.columns:
        vacios = (df[col] == "").sum()
        print(f"{col}: {vacios} registros vacíos después de limpieza")

nombre_producto: 0 registros vacíos después de limpieza
sinonimos: 35881 registros vacíos después de limpieza
uso: 16279 registros vacíos después de limpieza
fabricante: 1948 registros vacíos después de limpieza


In [9]:
# 8. Eliminar duplicados exactos si existen
df = df.drop_duplicates()
print("Dimensiones después de eliminar duplicados exactos:", df.shape)

Dimensiones después de eliminar duplicados exactos: (64495, 5)


In [10]:
# Separar el DataFrame en dos
# Uno con al menos un sinónimo o un uso
df_con_sin_o_uso = df[(df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != ''))]

# El otro con los demás
df_sin_sin_o_uso = df[~((df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != '')))]

print("DataFrame con sinónimo o uso:")
print(df_con_sin_o_uso.shape)
print("\nDataFrame sin sinónimo o uso:")
print(df_sin_sin_o_uso.shape)

DataFrame con sinónimo o uso:
(50621, 5)

DataFrame sin sinónimo o uso:
(13874, 5)


In [30]:
# 10. Construir columna de texto consolidado
def construir_texto_entrada(row):
    partes = [
        row.get("nombre_producto", ""),
        row.get("sinonimos", ""),
        row.get("uso", ""),
        row.get("fabricante", "")
    ]
    
    # Unir solo partes no vacías
    texto = " ".join([p for p in partes if isinstance(p, str) and p.strip() != ""])
    
    # Normalizar espacios por si acaso
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto

df["texto_entrada"] = df.apply(construir_texto_entrada, axis=1)

display(df[["id", "nombre_producto", "sinonimos", "uso", "fabricante", "texto_entrada"]].head(10))

,id,nombre_producto,sinonimos,uso,fabricante,texto_entrada
0,30818,0 1 m dtt,superscript iv first strand synthesis system k...,para uso en investigacion unicamente,thermofisher scientific,0 1 m dtt superscript iv first strand synthesi...
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,,beckman coulter,0 1n naoh compartimiento r1b access hstni b526...
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare,0 1n naoh cuvette cleaning solution b 7098 104...
3,42583,0 45 m 47mm white gridded s pak filter membran...,,investigacion y analisis bioquimicos,merck,0 45 m 47mm white gridded s pak filter membran...
4,56156,0 5 medios de crecimiento b de postgate modifi...,,,grupo volante,0 5 medios de crecimiento b de postgate modifi...
5,36322,0 ftu turbidity calibration standard,agua,patron de calibracion para medidores de turbidez,hanna instruments,0 ftu turbidity calibration standard agua patr...
6,66080,000000026354 hth ph minus,acido seco,regulador de ph y alcalinidad para piscinas,arch chemicals,000000026354 hth ph minus acido seco regulador...
7,43858,000000026354 hth ph minus,acido seco,regulador de ph y alcalinidad para piscinas,sigura,000000026354 hth ph minus acido seco regulador...
8,11166,000113 b iris,,,firmenich,000113 b iris firmenich
9,11160,000136 btr bergamote,,,firmenich,000136 btr bergamote firmenich


In [33]:
# 11. Versiones alternativas de texto

# Solo nombre + fabricante
df["texto_basico"] = (
    df["nombre_producto"].fillna("") + " " +
    df["fabricante"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

# Nombre + sinonimos  
df["texto_modelo"] = (
    df["nombre_producto"].fillna("") + " " +
    df["sinonimos"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

# Nombre + sinonimos + fabricante
df["texto_intermedio"] = (
    df["nombre_producto"].fillna("") + " " +
    df["sinonimos"].fillna("") + " " +
    df["fabricante"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

# Nombre + sinonimos + uso + fabricante
df["texto_completo"] = df["texto_entrada"]

display(df[["texto_basico", "texto_modelo", "texto_intermedio", "texto_completo"]].head(5))


,texto_basico,texto_modelo,texto_intermedio,texto_completo
0,0 1 m dtt thermofisher scientific,0 1 m dtt superscript iv first strand synthesi...,0 1 m dtt superscript iv first strand synthesi...,0 1 m dtt superscript iv first strand synthesi...
1,0 1n naoh compartimiento r1b beckman coulter,0 1n naoh compartimiento r1b access hstni b52699,0 1n naoh compartimiento r1b access hstni b526...,0 1n naoh compartimiento r1b access hstni b526...
2,0 1n naoh cuvette cleaning solution b siemens ...,0 1n naoh cuvette cleaning solution b 7098 104...,0 1n naoh cuvette cleaning solution b 7098 104...,0 1n naoh cuvette cleaning solution b 7098 104...
3,0 45 m 47mm white gridded s pak filter membran...,0 45 m 47mm white gridded s pak filter membran...,0 45 m 47mm white gridded s pak filter membran...,0 45 m 47mm white gridded s pak filter membran...
4,0 5 medios de crecimiento b de postgate modifi...,0 5 medios de crecimiento b de postgate modifi...,0 5 medios de crecimiento b de postgate modifi...,0 5 medios de crecimiento b de postgate modifi...


In [55]:
# 12. Resumen final de limpieza
print("Dimensiones finales:", df.shape)
print("\nEjemplos de texto consolidado:")
for i in range(5):
    print(f"\nRegistro {i+1}:")
    print(df.iloc[i]["texto_entrada"])

Dimensiones finales: (64495, 10)

Ejemplos de texto consolidado:

Registro 1:
0 1 m dtt superscript iv first strand synthesis system kit retrotranscriptasa iv sintesis cdna para uso en investigacion unicamente thermofisher scientific

Registro 2:
0 1n naoh compartimiento r1b access hstni b52699 beckman coulter

Registro 3:
0 1n naoh cuvette cleaning solution b 7098 10445578 agentes de diagn oacute stico siemens healthcare

Registro 4:
0 45 m 47mm white gridded s pak filter membrane made of mixed ester cellulose investigacion y analisis bioquimicos merck

Registro 5:
0 5 medios de crecimiento b de postgate modificados grupo volante


### Tokenización

In [35]:
# 14. Tokenización simple del texto consolidado

df_modelo = df.copy()

# Nos quedamos solo con registros que tengan texto en la representación principal
df_modelo = df_modelo[
    df_modelo["texto_modelo"].notna() &
    (df_modelo["texto_modelo"].str.strip() != "")
].copy()

# Tokenización básica por espacios
df_modelo["tokens"] = df_modelo["texto_modelo"].apply(lambda x: x.split())

print("Dimensiones del dataset para modelado:", df_modelo.shape)
display(df_modelo[["id", "texto_modelo", "tokens"]].head(5))


Dimensiones del dataset para modelado: (64495, 11)


,id,texto_modelo,tokens
0,30818,0 1 m dtt superscript iv first strand synthesi...,"[0, 1, m, dtt, superscript, iv, first, strand,..."
1,56997,0 1n naoh compartimiento r1b access hstni b52699,"[0, 1n, naoh, compartimiento, r1b, access, hst..."
2,10421,0 1n naoh cuvette cleaning solution b 7098 104...,"[0, 1n, naoh, cuvette, cleaning, solution, b, ..."
3,42583,0 45 m 47mm white gridded s pak filter membran...,"[0, 45, m, 47mm, white, gridded, s, pak, filte..."
4,56156,0 5 medios de crecimiento b de postgate modifi...,"[0, 5, medios, de, crecimiento, b, de, postgat..."


In [39]:
# 15. Longitud de los textos
df_modelo["num_tokens"] = df_modelo["tokens"].apply(len)

print("Resumen de longitud de textos:")
print(df_modelo["num_tokens"].describe())

print("\nPercentiles:")
print(df_modelo["num_tokens"].quantile([0.25, 0.5, 0.75, 0.9, 0.95]))

Resumen de longitud de textos:
count    64495.000000
mean         7.167858
std         11.834680
min          1.000000
25%          3.000000
50%          5.000000
75%          9.000000
max       1852.000000
Name: num_tokens, dtype: float64

Percentiles:
0.25     3.0
0.50     5.0
0.75     9.0
0.90    14.0
0.95    19.0
Name: num_tokens, dtype: float64


In [38]:
#!pip install gensim

In [40]:
# 16. Entrenamiento del modelo Word2Vec
from gensim.models import Word2Vec

# Corpus tokenizado
corpus_tokens = df_modelo["tokens"].tolist()

# Modelo Word2Vec
w2v_model = Word2Vec(
    sentences=corpus_tokens,
    vector_size=200,   # dimensión del embedding
    window=8,          # contexto
    min_count=2,       # ignora palabras muy raras
    workers=4,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    epochs=20
)

print("Tamaño del vocabulario aprendido:", len(w2v_model.wv))

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Tamaño del vocabulario aprendido: 26234


### Construir embedding por producto

In [56]:
# 16. Construir embedding por producto

import numpy as np

def vector_promedio_producto(tokens, model, vector_size=200):
    
    vectores = []

    for token in tokens:
        if token in model.wv:
            vectores.append(model.wv[token])

    if len(vectores) == 0:
        return np.zeros(vector_size)

    return np.mean(vectores, axis=0)


# Generar embedding de cada producto
embeddings_catalogo = np.vstack(
    df_modelo["tokens"].apply(
        lambda x: vector_promedio_producto(x, w2v_model, vector_size=200)
    )
)

print("Shape de embeddings del catálogo:", embeddings_catalogo.shape)

Shape de embeddings del catálogo: (64495, 200)


In [57]:
# 19. Tabla final del catálogo con embeddings
df_catalogo = df_modelo[["id", "nombre_producto", "sinonimos", "uso", "fabricante", "texto_modelo"]].copy()
df_catalogo["embedding_index"] = range(len(df_catalogo))

display(df_catalogo.head())

,id,nombre_producto,sinonimos,uso,fabricante,texto_modelo,embedding_index
0,30818,0 1 m dtt,superscript iv first strand synthesis system k...,para uso en investigacion unicamente,thermofisher scientific,0 1 m dtt superscript iv first strand synthesi...,0
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,,beckman coulter,0 1n naoh compartimiento r1b access hstni b52699,1
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare,0 1n naoh cuvette cleaning solution b 7098 104...,2
3,42583,0 45 m 47mm white gridded s pak filter membran...,,investigacion y analisis bioquimicos,merck,0 45 m 47mm white gridded s pak filter membran...,3
4,56156,0 5 medios de crecimiento b de postgate modifi...,,,grupo volante,0 5 medios de crecimiento b de postgate modifi...,4


### Clustering de productos (KMeans)

In [45]:
from sklearn.cluster import KMeans

k = 20  

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(embeddings_catalogo)

df_catalogo["cluster"] = clusters

display(df_catalogo[["id","nombre_producto","fabricante","cluster"]].head(10))


,id,nombre_producto,fabricante,cluster
0,30818,0 1 m dtt,thermofisher scientific,8
1,56997,0 1n naoh compartimiento r1b,beckman coulter,8
2,10421,0 1n naoh cuvette cleaning solution b,siemens healthcare,8
3,42583,0 45 m 47mm white gridded s pak filter membran...,merck,12
4,56156,0 5 medios de crecimiento b de postgate modifi...,grupo volante,15
5,36322,0 ftu turbidity calibration standard,hanna instruments,8
6,66080,000000026354 hth ph minus,arch chemicals,13
7,43858,000000026354 hth ph minus,sigura,13
8,11166,000113 b iris,firmenich,0
9,11160,000136 btr bergamote,firmenich,17


In [46]:
# 20. Analizar un cluster específico

cluster_id = 8   

cluster_muestra = df_catalogo[df_catalogo["cluster"] == cluster_id]

display(
    cluster_muestra[[
        "id",
        "nombre_producto",
        "fabricante",
        "sinonimos"
    ]].head(20)
)


,id,nombre_producto,fabricante,sinonimos
0,30818,0 1 m dtt,thermofisher scientific,superscript iv first strand synthesis system k...
1,56997,0 1n naoh compartimiento r1b,beckman coulter,access hstni b52699
2,10421,0 1n naoh cuvette cleaning solution b,siemens healthcare,7098 10445578
5,36322,0 ftu turbidity calibration standard,hanna instruments,agua
54,39590,003b glutaraldehido,aquaterra,
79,65327,0190001 bornyl acetate liquid,givaudan,
248,19240,1 10 fenantrolina anhidro,merck,1 10 fenantrolina anhidro para sintesis
249,48991,1 10 fenantrolina monohidratado 3 0 g l,brinsa,
250,17808,1 10 fenantrolina monohidrato p a acs,merck,1 10 fenantrolina monohidrato
251,19430,1 10 fenantrolinio cloruro monohidrato para an...,merck,


### Función para buscar productos similares

In [58]:
# 20. Función de similitud coseno
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def buscar_productos_similares(
    nombre_producto="",
    sinonimos="",
    fabricante="",
    top_k=5,
    filtrar_mismo_fabricante=False
):
    # Construir texto igual que en el entrenamiento 
    partes = [nombre_producto, sinonimos]
    partes = [limpiar_texto(p) for p in partes if p is not None]
    texto_consulta = " ".join([p for p in partes if p.strip() != ""]).strip()

    if texto_consulta == "":
        return pd.DataFrame(columns=["id", "nombre_producto", "fabricante", "texto_modelo", "similitud"])

    # Tokenizar
    tokens_consulta = texto_consulta.split()

    # Embedding de la consulta
    embedding_consulta = vector_promedio_producto(
        tokens_consulta,
        w2v_model,
        vector_size=200
    ).reshape(1, -1)

    # Similitud contra todo el catálogo
    sims = cosine_similarity(embedding_consulta, embeddings_catalogo)[0]

    # Armar resultados base
    resultados = df_catalogo.copy()
    resultados["similitud"] = sims

    # Regla opcional: filtrar por mismo fabricante
    if filtrar_mismo_fabricante and fabricante.strip() != "":
        fabricante_limpio = limpiar_texto(fabricante)
        resultados = resultados[
            resultados["fabricante"].fillna("").apply(limpiar_texto) == fabricante_limpio
        ].copy()

    # Ordenar por similitud y devolver top_k
    resultados = resultados.sort_values("similitud", ascending=False)

    return resultados[[
        "id",
        "nombre_producto",
        "fabricante",
        "texto_modelo",
        "similitud"
    ]].head(top_k).reset_index(drop=True)

In [59]:
# 21. Prueba funcional del modelo con una consulta nueva

resultados = buscar_productos_similares(
    nombre_producto="pegante base solvente",
    sinonimos="adhesivo industrial",
    fabricante="pegaucho",
    top_k=10,
    filtrar_mismo_fabricante=False
)

display(resultados)

,id,nombre_producto,fabricante,texto_modelo,similitud
0,58989,adhesivo base cloropreno,eterna,adhesivo base cloropreno pegantes amarillos,0.881836
1,57131,adhesivo industrial,incap,adhesivo industrial boxer,0.856690
2,37077,pegantes de policloropreno base solvente,eterna,pegantes de policloropreno base solvente pegan...,0.843660
3,23010,adhesivo industrial pliobond,ashland,adhesivo industrial pliobond,0.836463
4,2880,adhesivo maxon blanco,incap,adhesivo maxon blanco pegantes de poliuretano ...,0.835558
5,554,pegante industrial boxer,no registra,pegante industrial boxer,0.832771
6,15999,adhesivo industrial incofom,incap,adhesivo industrial incofom,0.823862
7,21608,pegante industrial colbon,ralph wilson plastics company,pegante industrial colbon adhesivo serie pva p...,0.811981
8,51086,solucion continental pegante solucion poliisop...,continental de pegantes y soluciones s a s,solucion continental pegante solucion poliisop...,0.807391
9,4530,pegante adhesivo base agua incafilm eva,incap,pegante adhesivo base agua incafilm eva pegant...,0.803075


In [61]:
# 21. Prueba con un registro existente del catálogo

ejemplo = df_catalogo.iloc[100]

print("Consulta:")
print("ID:", ejemplo["id"])
print("Nombre:", ejemplo["nombre_producto"])
print("Sinónimos:", ejemplo["sinonimos"])
print("Fabricante:", ejemplo["fabricante"])
print("\nTexto modelo:")
print(ejemplo["texto_modelo"])

print("\nTop 5 similares:")

display(
    buscar_productos_similares(
        nombre_producto=ejemplo["nombre_producto"],
        sinonimos=ejemplo["sinonimos"],
        fabricante=ejemplo["fabricante"],
        top_k=5
    )
)


# Prueba con una consulta externa
resultados = buscar_productos_similares(
    nombre_producto="pegante base solvente",
    sinonimos="adhesivo",
    fabricante="pegaucho",
    top_k=5
)

display(resultados)


Consulta:
ID: 11365
Nombre: 039576 b fenouil doux subst
Sinónimos: 
Fabricante: firmenich

Texto modelo:
039576 b fenouil doux subst

Top 5 similares:


,id,nombre_producto,fabricante,texto_modelo,similitud
0,11365,039576 b fenouil doux subst,firmenich,039576 b fenouil doux subst,1.000000
1,53808,opticlean b,h2o innovation pwt chemicals,opticlean b pc37,0.992052
2,30866,nica b,yara,nica b plan1g,0.991354
3,34024,amphotericin b,sigma aldrich,amphotericin b phr1662,0.990815
4,58271,qv b,qualipoly chemical corporation,qv b,0.990650


,id,nombre_producto,fabricante,texto_modelo,similitud
0,58989,adhesivo base cloropreno,eterna,adhesivo base cloropreno pegantes amarillos,0.908368
1,37077,pegantes de policloropreno base solvente,eterna,pegantes de policloropreno base solvente pegan...,0.849101
2,2880,adhesivo maxon blanco,incap,adhesivo maxon blanco pegantes de poliuretano ...,0.842401
3,39373,superfachada base solvente,corlanc,superfachada base solvente,0.826271
4,62799,quimeroil ozas azaxolidina base solvente,quimeria,quimeroil ozas azaxolidina base solvente,0.821774


### Evaluación simple del sistema

In [27]:
# 23. Evaluación rápida
def evaluar_recall_at_k(df_eval, top_k=5, n_muestras=200):
    indices = np.random.choice(len(df_eval), size=min(n_muestras, len(df_eval)), replace=False)
    aciertos = 0

    for idx in indices:
        fila = df_eval.iloc[idx]

        resultados = buscar_productos_similares(
            nombre_producto=fila["nombre_producto"],
            sinonimos=df_modelo.iloc[idx]["sinonimos"],
            uso=df_modelo.iloc[idx]["uso"],
            fabricante=fila["fabricante"],
            top_k=top_k
        )

        resultados = resultados[resultados["id"].astype(str) != str(fila["id"])]

        if str(fila["id"]) in resultados["id"].astype(str).values:
            aciertos += 1

    recall_k = aciertos / len(indices)
    return recall_k


recall_5 = evaluar_recall_at_k(df_catalogo, top_k=5, n_muestras=200)
print(f"Recall@5 aproximado: {recall_5:.4f}")

Recall@5 aproximado: 0.0000


In [28]:
# 24. Precisión aproximada
def evaluar_precision_at_k(df_eval, top_k=5, n_muestras=200):
    indices = np.random.choice(len(df_eval), size=min(n_muestras, len(df_eval)), replace=False)
    precisiones = []

    for idx in indices:
        fila = df_eval.iloc[idx]

        resultados = buscar_productos_similares(
            nombre_producto=fila["nombre_producto"],
            sinonimos=df_modelo.iloc[idx]["sinonimos"],
            uso=df_modelo.iloc[idx]["uso"],
            fabricante=fila["fabricante"],
            top_k=top_k
        )

        resultados = resultados[resultados["id"].astype(str) != str(fila["id"])]

        aciertos = (resultados["id"].astype(str) == str(fila["id"])).sum()

        precision_k = aciertos / top_k
        precisiones.append(precision_k)

    return np.mean(precisiones)


precision_5 = evaluar_precision_at_k(df_catalogo, top_k=5, n_muestras=200)
print(f"Precisión@5 aproximada: {precision_5:.4f}")

Precisión@5 aproximada: 0.0000
